# W10 Lab: zero-shot stance detection with NLI

A short companion to the integrated Week 10 guide. Adapted from Michael Burnham's official zero-shot stance tutorial using three constructed classroom texts.


## What changes from supervised classification?

The earlier Flair and Transformers notebooks learn environmental-policy labels from human-coded training examples. This exercise instead reuses a Transformer already trained on natural language inference (NLI). We supply the stance meanings as complete hypotheses. This is a separate Trump stance task: the environmental-policy classifier's outputs cannot simply be renamed.

You can run this notebook independently, without fitting either policy model.


## Packages and device

Use the prepared Python environment with `torch`, `transformers`, `pandas` and `sentencepiece`. In a fresh notebook environment, run the following installation command once in the active kernel, then restart it:

```python
%pip install torch transformers pandas sentencepiece
```

The model-loading cell may download a large checkpoint. This demonstration explicitly uses CPU (`device=-1`), so it can take time. The author's [Colab tutorial](https://colab.research.google.com/drive/1YLmI2WMkIVpgeUobgMDVXkjf8JfbRxvW?usp=sharing) is another way to follow the example.


In [ ]:
import pandas as pd
import torch
from transformers import pipeline


## N1. Load an existing NLI model

`pipeline("zero-shot-classification", ...)` uses a model trained to recognize relations between a text and a hypothesis. Loading this checkpoint does not train a new Trump stance classifier. Keep its complete model ID with your results.

The checkpoint revision is pinned to the displayed CPU demonstration. Explicit float32 precision and batch size one match that run.


In [ ]:
nli_model_id = (
    "MoritzLaurer/"
    "DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
)
nli_revision = "b3546ea6b0346eb6f8d5d68b13c7dc6d0376b3d7"
nli_classifier = pipeline(
    "zero-shot-classification",
    model=nli_model_id,
    revision=nli_revision,
    device=-1,
    torch_dtype=torch.float32,
    batch_size=1,
)


**Check:** which earlier call updated the policy classifier's weights? Why is there no `Trainer` or `.train()` call here? Zero-shot means no target-task parameter updates in this exercise; the NLI model has already been trained.


## N2. Make the target and categories explicit

The template supplies the target, Trump. Each candidate fills the blank to create a complete hypothesis. Print these statements before looking at predictions.


In [ ]:
nli_labels = [
    "supports",
    "opposes",
    "does not express an opinion about",
]
nli_template = "The author of this tweet {} Trump."

nli_hypotheses = pd.DataFrame({
    "candidate": nli_labels,
    "hypothesis": [
        nli_template.format(label)
        for label in nli_labels
    ],
})
print(nli_hypotheses.to_string(index=False))


                        candidate                                                        hypothesis
                         supports                          The author of this tweet supports Trump.
                          opposes                           The author of this tweet opposes Trump.
does not express an opinion about The author of this tweet does not express an opinion about Trump.


**Read the constructed hypotheses:**

| Candidate | Complete hypothesis |
|---|---|
| supports | The author of this tweet supports Trump. |
| opposes | The author of this tweet opposes Trump. |
| does not express an opinion about | The author of this tweet does not express an opinion about Trump. |

The model will pair each text with all three statements. These strings are inputs we chose, not labels or explanations generated by the model.

**Check:** what would need to change to study stance toward a bill?


## N3. Read three texts before running the classifier

Texts A and B explicitly support or oppose Trump's policies. Text C states a meeting time without expressing a stance toward Trump. The `reading_key` records our intended reading of a constructed example. It is not a human annotation from a research corpus, and these three texts are not an accuracy benchmark.


In [ ]:
nli_texts = [
    "I support Trump's policies.",
    "I oppose Trump's policies.",
    "The committee meets on Tuesday.",
]
nli_examples = pd.DataFrame({
    "example": ["A", "B", "C"],
    "text": nli_texts,
    "reading_key": [
        "supports",
        "opposes",
        "does not express an opinion about",
    ],
})


Pair the texts with the candidate hypotheses. `multi_label=False` ranks the candidates as mutually exclusive stance choices for each text. The same input text is evaluated against each complete hypothesis.


In [ ]:
nli_results = nli_classifier(
    nli_texts,
    candidate_labels=nli_labels,
    hypothesis_template=nli_template,
    multi_label=False,
)


## Keep scores attached to candidate names

The pipeline returns candidates in score order, which can differ by text. Convert each returned label-score pair to a dictionary before making a table. Print the full text, reading key, predicted stance and all three scores.


In [ ]:
nli_rows = []
for example, result in zip(
    nli_examples.to_dict("records"), nli_results
):
    scores = dict(zip(result["labels"], result["scores"]))
    nli_rows.append({
        **example,
        "predicted_stance": result["labels"][0],
        **{label: scores[label] for label in nli_labels},
    })
nli_table = pd.DataFrame(nli_rows)
print(nli_table.to_string(
    index=False,
    float_format=lambda value: f"{value:.6f}",
))


example                            text                       reading_key                  predicted_stance  supports  opposes  does not express an opinion about
      A     I support Trump's policies.                          supports                          supports  0.991822 0.004377                           0.003801
      B      I oppose Trump's policies.                           opposes                           opposes  0.008588 0.984321                           0.007091
      C The committee meets on Tuesday. does not express an opinion about does not express an opinion about  0.206294 0.196707                           0.597000


**Executed demonstration · 27 November 2026**

The saved table output comes from a real CPU inference run with PyTorch 2.2.2, Transformers 4.54.0 and the pinned checkpoint revision. No target-task training occurred. The code cells start without execution counters; rerun them to generate your own outputs. The saved printout records the dated demonstration, not a new research validation set.

Read A, then B, then C. Explain each top label using the text. A receives supports at .991822; B receives opposes at .984321. C receives the no-opinion candidate at .597000, with supports at .206294 and opposes at .196707. The winning candidate is not certainty. The reading keys are constructed teaching interpretations, not annotations from a research corpus.

With `multi_label=False`, the pipeline normalizes candidate entailment logits across the candidate set. These scores are **not accuracy** and are **not the full entailment/neutral/contradiction distribution**. Changing the candidates can change the scores. The explicit “does not express an opinion” stance is different from NLI `neutral`, which means a particular hypothesis remains unresolved by its text.

[Read the complete executed printout](../results/burnham-nli-demo.txt).


## N4. Compare the workflows

| Question | Earlier supervised classification | This zero-shot NLI exercise |
|---|---|---|
| Input | Policy excerpt | Text paired with complete stance hypotheses |
| Research labels | Human labels train a task-specific classifier | Category meanings are written as hypotheses |
| Weight updates | The training call updates weights | The existing NLI checkpoint is reused |
| Validation | Evaluate predictions on held-out labeled texts | Evaluate hypotheses and the decision rule on target-task human labels |

Both use Transformers. NLI models can also be fine-tuned; this exercise uses one without target-task updates. The three classroom examples do not validate a research measure.


## Check and submit

Submit the printed three-text table and explain one prediction. Propose one alternative hypothesis wording and describe how you would test its effect on separately human-coded target texts.

1. Why is the hypothesis part of the input?
2. Why cannot policy categories simply be renamed as stance categories?
3. How does the “does not express an opinion” candidate differ from NLI `neutral`?
4. What evidence would you need before applying these hypotheses to a research corpus?


## Sources

- [Integrated Week 10 Lab guide](../Lab_Session_W10.html#nli-demo)
- [Burnham's official zero-shot notebook](https://github.com/MLBurnham/stance_detection_tutorials/blob/main/Zero-shot_stance_detection.ipynb)
- [Official Colab](https://colab.research.google.com/drive/1YLmI2WMkIVpgeUobgMDVXkjf8JfbRxvW?usp=sharing)
- [DeBERTa-v3-large NLI model card](https://huggingface.co/MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli)
- [Hugging Face ZeroShotClassificationPipeline](https://huggingface.co/docs/transformers/en/main_classes/pipelines#transformers.ZeroShotClassificationPipeline)

The tutorial demonstrates Trump stance. Burnham's paper separately applies topic-matched hypotheses to COVID-19 tweets. This small notebook does not reproduce that full analysis.
